<a href="https://colab.research.google.com/github/Fatou-Kine3/github_task/blob/main/Recurrent_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Problem 1

In [1]:
import numpy as np


def simple_rnn_forward(x, w_x, w_h, b, h_0=None, return_all_states=False):
    """
    Forward propagation for a simple RNN.

    Parameters
    ----------
    x : np.ndarray, shape (batch_size, n_sequences, n_features)
        Full input sequence.
    w_x : np.ndarray, shape (n_features, n_nodes)
        Input-to-hidden weights.
    w_h : np.ndarray, shape (n_nodes, n_nodes)
        Hidden-to-hidden (recurrent) weights.
    b : np.ndarray, shape (n_nodes,)
        Bias vector.
    h_0 : np.ndarray or None, shape (batch_size, n_nodes)
        Initial hidden state. If None, zeros are used.
    return_all_states : bool
        If True, also return the list of all hidden states
        [h_0, h_1, ..., h_T].

    Returns
    -------
    h_T : np.ndarray, shape (batch_size, n_nodes)
        Final hidden state.
    states : list of np.ndarray, optional
        All hidden states (only if return_all_states=True).
    """
    batch_size, n_sequences, n_features = x.shape
    n_nodes = w_x.shape[1]

    if h_0 is None:
        h_0 = np.zeros((batch_size, n_nodes), dtype=x.dtype)

    h_prev = h_0
    states = [h_prev]

    for t in range(n_sequences):
        x_t = x[:, t, :]                          # (batch_size, n_features)
        a_t = x_t @ w_x + h_prev @ w_h + b        # (batch_size, n_nodes)
        h_t = np.tanh(a_t)                        # (batch_size, n_nodes)

        states.append(h_t)
        h_prev = h_t

    h_T = h_prev

    if return_all_states:
        return h_T, states
    return h_T

In [ ]:
## Problem 2

In [2]:
import numpy as np

x = np.array([[[1, 2], [2, 3], [3, 4]]], dtype=np.float64) / 100
w_x = np.array(
    [[1, 3, 5, 7], [3, 5, 7, 8]],
    dtype=np.float64,
) / 100
w_h = np.array(
    [[1, 3, 5, 7], [2, 4, 6, 8], [3, 5, 7, 8], [4, 6, 8, 10]],
    dtype=np.float64,
) / 100
b = np.array([1, 1, 1, 1], dtype=np.float64)
h_0 = np.zeros((x.shape[0], w_x.shape[1]), dtype=np.float64)

h_T, states = simple_rnn_forward(x, w_x, w_h, b, h_0, return_all_states=True)

print("Final hidden state:")
print(h_T)
print()

print("State at every position:")
for t, h_t in enumerate(states):
    print(f"h_{t} = {h_t}")

# Verify the final state
expected = np.array(
    [[0.79494228, 0.81839002, 0.83939649, 0.85584174]]
)
np.testing.assert_allclose(h_T, expected, rtol=1e-7, atol=1e-7)
print("\nFinal hidden state matches expected values.")

Final hidden state:
[[0.79494228 0.81839002 0.83939649 0.85584174]]

State at every position:
h_0 = [[0. 0. 0. 0.]]
h_1 = [[0.76188798 0.76213958 0.76239095 0.76255841]]
h_2 = [[0.792209   0.8141834  0.83404912 0.84977719]]
h_3 = [[0.79494228 0.81839002 0.83939649 0.85584174]]

Final hidden state matches expected values.


In [ ]:
## Problem 3

In [3]:
import numpy as np


def simple_rnn_forward_with_cache(x, w_x, w_h, b, h_0=None):
    """
    Forward propagation that also caches everything needed for BPTT.

    Returns
    -------
    h_T : np.ndarray, shape (batch_size, n_nodes)
    cache : dict with keys
        'x'      : input (batch_size, n_sequences, n_features)
        'h'      : list [h_0, h_1, ..., h_T]
        'a'      : list [a_1, ..., a_T]
        'w_x', 'w_h', 'b'
    """
    batch_size, n_sequences, n_features = x.shape
    n_nodes = w_x.shape[1]

    if h_0 is None:
        h_0 = np.zeros((batch_size, n_nodes), dtype=x.dtype)

    h_prev = h_0
    h_list = [h_prev]
    a_list = []

    for t in range(n_sequences):
        x_t = x[:, t, :]
        a_t = x_t @ w_x + h_prev @ w_h + b
        h_t = np.tanh(a_t)

        a_list.append(a_t)
        h_list.append(h_t)
        h_prev = h_t

    cache = {
        "x": x,
        "h": h_list,
        "a": a_list,
        "w_x": w_x,
        "w_h": w_h,
        "b": b,
    }
    return h_prev, cache


def simple_rnn_bptt(cache, dh_T):
    """
    Backpropagation through time for the simple RNN.

    Parameters
    ----------
    cache : dict
        Output of simple_rnn_forward_with_cache.
    dh_T : np.ndarray, shape (batch_size, n_nodes)
        Gradient of the loss w.r.t. the final hidden state h_T.

    Returns
    -------
    grads : dict with keys 'w_x', 'w_h', 'b', 'h_0'
        Gradients w.r.t. the parameters and the initial hidden state.
    """
    x = cache["x"]
    h_list = cache["h"]          # [h_0, h_1, ..., h_T]
    a_list = cache["a"]          # [a_1, ..., a_T]
    w_x = cache["w_x"]
    w_h = cache["w_h"]
    b = cache["b"]

    batch_size, n_sequences, n_features = x.shape
    n_nodes = w_x.shape[1]

    dw_x = np.zeros_like(w_x)
    dw_h = np.zeros_like(w_h)
    db = np.zeros_like(b)

    dh_next = dh_T.copy()        # gradient arriving at h_t from the future

    for t in reversed(range(n_sequences)):
        h_t = h_list[t + 1]      # h_t
        h_prev = h_list[t]       # h_{t-1}
        x_t = x[:, t, :]
        a_t = a_list[t]

        # Total gradient at h_t
        dh_t = dh_next

        # Through tanh
        da_t = dh_t * (1.0 - h_t ** 2)          # (batch_size, n_nodes)

        # Parameter gradients (accumulate over time)
        dw_x += x_t.T @ da_t                     # (n_features, n_nodes)
        dw_h += h_prev.T @ da_t                  # (n_nodes, n_nodes)
        db += da_t.sum(axis=0)                   # (n_nodes,)

        # Gradient to previous hidden state
        dh_prev = da_t @ w_h.T                   # (batch_size, n_nodes)

        # Gradient to input (not needed for parameter updates, but shown)
        # dx_t = da_t @ w_x.T

        dh_next = dh_prev

    grads = {
        "w_x": dw_x,
        "w_h": dw_h,
        "b": db,
        "h_0": dh_next,   # gradient w.r.t. initial state
    }
    return grads


def simple_rnn_update(w_x, w_h, b, grads, lr):
    """Apply one SGD update."""
    w_x = w_x - lr * grads["w_x"]
    w_h = w_h - lr * grads["w_h"]
    b = b - lr * grads["b"]
    return w_x, w_h, b

In [4]:
def numerical_gradient(f, param, eps=1e-6):
    """Central finite-difference gradient of scalar f w.r.t. array param."""
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=["multi_index"], order="C")
    while not it.finished:
        idx = it.multi_index
        original = param[idx]

        param[idx] = original + eps
        f_plus = f()

        param[idx] = original - eps
        f_minus = f()

        param[idx] = original
        grad[idx] = (f_plus - f_minus) / (2 * eps)
        it.iternext()
    return grad


def run_gradient_check():
    rng = np.random.default_rng(0)

    batch_size, n_sequences, n_features, n_nodes = 1, 3, 2, 3

    x = rng.normal(size=(batch_size, n_sequences, n_features))
    w_x = rng.normal(size=(n_features, n_nodes))
    w_h = rng.normal(size=(n_nodes, n_nodes))
    b = rng.normal(size=(n_nodes,))
    h_0 = rng.normal(size=(batch_size, n_nodes))
    y = rng.normal(size=(batch_size, n_nodes))   # target for h_T

    # Loss: 0.5 * sum((h_T - y)^2)
    def loss_fn():
        h_T, _ = simple_rnn_forward_with_cache(x, w_x, w_h, b, h_0)
        return 0.5 * np.sum((h_T - y) ** 2)

    # Analytical gradients
    h_T, cache = simple_rnn_forward_with_cache(x, w_x, w_h, b, h_0)
    dh_T = h_T - y
    grads = simple_rnn_bptt(cache, dh_T)

    # Numerical gradients
    num_w_x = numerical_gradient(loss_fn, w_x)
    num_w_h = numerical_gradient(loss_fn, w_h)
    num_b = numerical_gradient(loss_fn, b)
    num_h_0 = numerical_gradient(loss_fn, h_0)

    # Compare
    for name, ana, num in [
        ("w_x", grads["w_x"], num_w_x),
        ("w_h", grads["w_h"], num_w_h),
        ("b", grads["b"], num_b),
        ("h_0", grads["h_0"], num_h_0),
    ]:
        np.testing.assert_allclose(
            ana, num, rtol=1e-5, atol=1e-7,
            err_msg=f"Gradient mismatch for {name}",
        )
        print(f"{name}: analytical and numerical gradients match.")

    print("\nAll gradient checks passed.")


if __name__ == "__main__":
    run_gradient_check()

w_x: analytical and numerical gradients match.
w_h: analytical and numerical gradients match.
b: analytical and numerical gradients match.
h_0: analytical and numerical gradients match.

All gradient checks passed.


In [5]:
class ScratchSimpleRNNClassifier:
    """
    Simple RNN followed by a linear classifier head.
    Uses only NumPy for the recurrent part.
    """

    def __init__(self, n_features, n_nodes, n_classes, seed=0):
        rng = np.random.default_rng(seed)
        self.w_x = rng.normal(scale=0.1, size=(n_features, n_nodes))
        self.w_h = rng.normal(scale=0.1, size=(n_nodes, n_nodes))
        self.b = np.zeros(n_nodes)
        self.w_o = rng.normal(scale=0.1, size=(n_nodes, n_classes))
        self.b_o = np.zeros(n_classes)
        self.h_0 = np.zeros((1, n_nodes))

    def forward(self, x):
        batch_size = x.shape[0]
        h_0 = np.zeros((batch_size, self.w_x.shape[1]))
        h_T, cache = simple_rnn_forward_with_cache(
            x, self.w_x, self.w_h, self.b, h_0
        )
        logits = h_T @ self.w_o + self.b_o
        cache["h_T"] = h_T
        cache["logits"] = logits
        return logits, cache

    @staticmethod
    def softmax(logits):
        shifted = logits - logits.max(axis=1, keepdims=True)
        exp = np.exp(shifted)
        return exp / exp.sum(axis=1, keepdims=True)

    def loss(self, logits, y):
        """Cross-entropy loss. y is integer class labels."""
        probs = self.softmax(logits)
        n = y.shape[0]
        return -np.log(probs[np.arange(n), y] + 1e-12).mean()

    def backward(self, cache, y):
        """Compute gradients for all parameters."""
        h_T = cache["h_T"]
        logits = cache["logits"]
        probs = self.softmax(logits)

        n = y.shape[0]
        dlogits = probs.copy()
        dlogits[np.arange(n), y] -= 1.0
        dlogits /= n                                 # (batch, n_classes)

        dw_o = h_T.T @ dlogits                       # (n_nodes, n_classes)
        db_o = dlogits.sum(axis=0)                   # (n_classes,)
        dh_T = dlogits @ self.w_o.T                  # (batch, n_nodes)

        grads = simple_rnn_bptt(cache, dh_T)
        grads["w_o"] = dw_o
        grads["b_o"] = db_o
        return grads

    def update(self, grads, lr=1e-2):
        self.w_x -= lr * grads["w_x"]
        self.w_h -= lr * grads["w_h"]
        self.b -= lr * grads["b"]
        self.w_o -= lr * grads["w_o"]
        self.b_o -= lr * grads["b_o"]

In [6]:
def train_tiny_task():
    rng = np.random.default_rng(42)
    batch_size, n_sequences, n_features, n_nodes = 8, 5, 3, 4

    x = rng.normal(size=(batch_size, n_sequences, n_features))
    w_x = rng.normal(scale=0.1, size=(n_features, n_nodes))
    w_h = rng.normal(scale=0.1, size=(n_nodes, n_nodes))
    b = np.zeros(n_nodes)
    h_0 = np.zeros((batch_size, n_nodes))

    # Target: a fixed function of the sequence, e.g. sum over time
    y = np.tanh(x.sum(axis=1)) @ rng.normal(size=(n_features, n_nodes))

    lr = 0.1
    for step in range(200):
        h_T, cache = simple_rnn_forward_with_cache(x, w_x, w_h, b, h_0)
        loss = 0.5 * np.sum((h_T - y) ** 2)
        dh_T = h_T - y
        grads = simple_rnn_bptt(cache, dh_T)
        w_x, w_h, b = simple_rnn_update(w_x, w_h, b, grads, lr)

        if step % 20 == 0:
            print(f"step {step:3d}  loss {loss:.6f}")

train_tiny_task()

step   0  loss 30.149069
step  20  loss 19.887914
step  40  loss 11.791680
step  60  loss 12.146895
step  80  loss 27.584106
step 100  loss 14.917297
step 120  loss 13.155741
step 140  loss 12.256584
step 160  loss 12.468493
step 180  loss 11.751812


In [7]:
def train_classifier_demo():
    rng = np.random.default_rng(0)
    n_samples, n_sequences, n_features, n_classes = 200, 6, 2, 2

    X = rng.normal(size=(n_samples, n_sequences, n_features))
    # Label = 1 if the first feature's sum over time exceeds 0
    y = (X[:, :, 0].sum(axis=1) > 0).astype(int)

    clf = ScratchSimpleRNNClassifier(n_features, n_nodes=5, n_classes=n_classes)

    for epoch in range(50):
        logits, cache = clf.forward(X)
        loss = clf.loss(logits, y)
        grads = clf.backward(cache, y)
        clf.update(grads, lr=1e-2)

        if epoch % 5 == 0:
            acc = (logits.argmax(axis=1) == y).mean()
            print(f"epoch {epoch:2d}  loss {loss:.4f}  acc {acc:.3f}")

train_classifier_demo()

epoch  0  loss 0.6957  acc 0.410
epoch  5  loss 0.6955  acc 0.410
epoch 10  loss 0.6952  acc 0.425
epoch 15  loss 0.6950  acc 0.425
epoch 20  loss 0.6949  acc 0.455
epoch 25  loss 0.6947  acc 0.455
epoch 30  loss 0.6945  acc 0.490
epoch 35  loss 0.6943  acc 0.505
epoch 40  loss 0.6941  acc 0.505
epoch 45  loss 0.6939  acc 0.505
